In [1]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

In [2]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])

orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

In [5]:
orders["delivery_days"].isnull().sum()

np.int64(2965)

In [4]:
orders_clean = orders.dropna(subset=["delivery_days"])
orders_clean.shape

(96476, 9)

2,965 orders (~3%) have no delivery_days value because order_delivered_customer_date 
is missing — these orders were never delivered (likely cancelled or lost). Since 
there's no real underlying delivery time to estimate for an order that was never 
delivered, imputing a value would fabricate data. Decision: remove these rows from 
the modeling dataset via dropna().

In [6]:
orders_clean["is_outlier_delivery"] = orders_clean["delivery_days"] > 60
orders_clean["is_outlier_delivery"].sum()

np.int64(288)

In [7]:
orders_clean.duplicated().sum()
items.duplicated().sum()
customers.duplicated().sum()
sellers.duplicated().sum()
products.duplicated().sum()

np.int64(0)

In [8]:
orders_clean["order_approved_at"] = pd.to_datetime(orders_clean["order_approved_at"])
orders_clean["order_delivered_carrier_date"] = pd.to_datetime(orders_clean["order_delivered_carrier_date"])
orders_clean["order_estimated_delivery_date"] = pd.to_datetime(orders_clean["order_estimated_delivery_date"])

In [9]:
orders_clean.info()

<class 'pandas.DataFrame'>
Index: 96476 entries, 0 to 99440
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96476 non-null  str           
 1   customer_id                    96476 non-null  str           
 2   order_status                   96476 non-null  str           
 3   order_purchase_timestamp       96476 non-null  datetime64[us]
 4   order_approved_at              96462 non-null  datetime64[us]
 5   order_delivered_carrier_date   96475 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  96476 non-null  datetime64[us]
 8   delivery_days                  96476 non-null  float64       
 9   is_outlier_delivery            96476 non-null  bool          
dtypes: bool(1), datetime64[us](5), float64(1), str(3)
memory usage: 7.5 MB


In [10]:
orders_clean["is_late"] = orders_clean["order_delivered_customer_date"] > orders_clean["order_estimated_delivery_date"]
orders_clean["is_late"].sum()

np.int64(7827)

In [11]:
orders_clean["purchase_month"] = orders_clean["order_purchase_timestamp"].dt.month
orders_clean["purchase_dayofweek"] = orders_clean["order_purchase_timestamp"].dt.dayofweek

In [12]:
orders_clean[["purchase_month", "purchase_dayofweek"]].head()

,purchase_month,purchase_dayofweek
0,10,0
1,7,1
2,8,2
3,11,5
4,2,1


In [13]:
orders_clean.groupby("purchase_month")["is_late"].mean()

purchase_month
1     0.062284
2     0.134243
3     0.171536
4     0.059554
5     0.066446
6     0.022099
7     0.040786
8     0.075778
9     0.052277
10    0.050548
11    0.143112
12    0.083787
Name: is_late, dtype: float64

In [14]:
orders_clean["purchase_month"].value_counts().sort_index()

purchase_month
1      7819
2      8209
3      9549
4      9101
5     10294
6      9231
7     10028
8     10544
9      4151
10     4748
11     7288
12     5514
Name: count, dtype: int64

## Seasonal Late-Delivery Pattern

Late delivery rate varies significantly by month, ranging from 2.2% (June) to 17.2% 
(March). November also shows an elevated rate (14.3%), plausibly linked to Black 
Friday/Cyber Monday order surges. February and March's high rates may relate to 
Brazilian Carnival season, though this requires further verification against exact 
holiday dates. June's low rate is backed by a normal sample size (9,231 orders), 
confirming it's a genuine pattern, not statistical noise from a small sample.

In [16]:
writer = pd.ExcelWriter("../reports/cleaning_report.xlsx", engine="openpyxl")